# Deep Past: Akkadian-English Translation - Kaggle Training Notebook

This notebook implements the 3-stage curriculum learning pipeline for training an Akkadian to English translation model.

## Pipeline Overview
1. **Stage 1**: Pre-training on general Akkadian corpus
2. **Stage 2**: Domain adaptation on OARE data
3. **Stage 3**: Competition fine-tuning with genre conditioning

## Setup Instructions
1. Add the competition dataset to this notebook
2. Upload your data files or add them as a Kaggle dataset
3. Enable GPU (P100 or T4 recommended)
4. Run all cells

## 1. Environment Setup

In [ ]:
# Install required packages (Kaggle has most pre-installed)
!pip install -q transformers>=4.41.0 sentencepiece sacrebleu evaluate datasets peft sacremoses

# Disable tokenizers parallelism warning
import os
os.environ['TOKENIZERS_PARALLELISM'] = 'false'

### [OPTIONAL] Fix CUDA Compatibility

If you see `AcceleratorError: CUDA error: no kernel image is available`, run the cell below to reinstall PyTorch with the correct CUDA version. **Only run this if the CUDA test fails.**

In [ ]:
# OPTIONAL: Uncomment and run this cell ONLY if CUDA test fails
# This will reinstall PyTorch with CUDA 11.8 support

# import subprocess
# import sys
# 
# print("Reinstalling PyTorch with CUDA 11.8 support...")
# subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
#     "torch==2.1.0+cu118", "torchvision==0.16.0+cu118", "torchaudio==2.1.0+cu118",
#     "--extra-index-url", "https://download.pytorch.org/whl/cu118"])
# 
# print("Done! Please restart the kernel and run from the beginning.")

In [ ]:
import os
import sys
import json
import random
from pathlib import Path
from datetime import datetime, timezone
from functools import partial

import numpy as np
import pandas as pd
import torch
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from transformers import (
    AutoModelForSeq2SeqLM,
    AutoTokenizer,
    get_linear_schedule_with_warmup,
)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

USE_GPU = False

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {gpu_name}")
    print(f"GPU Memory: {gpu_mem:.1f} GB")
    print(f"CUDA version: {torch.version.cuda}")
    
    # Test CUDA with tensor operations that mimic actual training
    print("\nTesting CUDA compatibility...")
    try:
        # Test 1: Basic tensor creation
        test_tensor = torch.zeros(1).cuda()
        print("  [OK] Basic tensor creation")
        
        # Test 2: Matrix multiplication (this is what fails with incompatible CUDA)
        a = torch.randn(64, 256).cuda()
        b = torch.randn(256, 128).cuda()
        c = torch.matmul(a, b)
        print("  [OK] Matrix multiplication")
        
        # Test 3: masked_fill_ operation (this is what actually failed)
        test = torch.zeros(10, 10).cuda()
        test.masked_fill_(test == 0, 1)
        print("  [OK] Masked fill operation")
        
        # All tests passed
        USE_GPU = True
        print("\n[SUCCESS] CUDA is fully compatible! Using GPU.")
        
    except Exception as e:
        print(f"\n[FAILED] CUDA test failed: {e}")
        print("\nThe GPU is not compatible with the installed PyTorch version.")
        print("Falling back to CPU training (this will be slower).")
        USE_GPU = False
        # Clear CUDA memory
        torch.cuda.empty_cache()
else:
    print("No GPU available, using CPU.")

# Set device based on test results
DEVICE = torch.device("cuda" if USE_GPU else "cpu")
print(f"\nSelected device: {DEVICE}")

In [ ]:
# ============================================================================
# CONFIGURATION
# ============================================================================

# Check if running on Kaggle
RUNNING_ON_KAGGLE = os.path.exists("/kaggle/input") or "KAGGLE_KERNEL_RUN_TYPE" in os.environ

if RUNNING_ON_KAGGLE:
    # Kaggle paths
    INPUT_BASE = Path("/kaggle/input")
    WORKING_DIR = Path("/kaggle/working")
    
    # List all files recursively to help debug
    print("Scanning /kaggle/input for CSV files...")
    all_csvs = list(INPUT_BASE.rglob("*.csv"))
    print(f"Found {len(all_csvs)} CSV files:")
    for csv in all_csvs[:20]:  # Show first 20
        print(f"  {csv}")
    if len(all_csvs) > 20:
        print(f"  ... and {len(all_csvs) - 20} more")
    
    # Find stage1_train.csv and get its parent directory
    DATA_DIR = None
    for csv_file in INPUT_BASE.rglob("stage1_train.csv"):
        DATA_DIR = csv_file.parent
        print(f"\nFound stage1_train.csv at: {csv_file}")
        break
    
    if DATA_DIR is None:
        raise FileNotFoundError(
            "Could not find stage1_train.csv in any dataset. "
            "Make sure your dataset contains the training CSV files."
        )
    
    OUTPUT_DIR = WORKING_DIR / "models"
else:
    # Local development paths
    PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "kaggle" else Path.cwd()
    DATA_DIR = PROJECT_ROOT / "data"
    OUTPUT_DIR = PROJECT_ROOT / "models"

# Create output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"\nRunning on Kaggle: {RUNNING_ON_KAGGLE}")
print(f"Data directory: {DATA_DIR}")
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
# Verify data files exist
REQUIRED_FILES = {
    "stage1_train": DATA_DIR / "stage1_train.csv",
    "stage2_train": DATA_DIR / "stage2_train.csv",
    "stage3_train": DATA_DIR / "stage3_train.csv",
    "stage3_val": DATA_DIR / "stage3_val.csv",
}

print("Checking data files:")
all_found = True
for name, path in REQUIRED_FILES.items():
    exists = path.exists()
    status = "OK" if exists else "MISSING"
    print(f"  {name}: {status}")
    if not exists:
        all_found = False

if not all_found:
    print("\n[ERROR] Some data files are missing!")
    print("\nContents of DATA_DIR:")
    if DATA_DIR.exists():
        for item in DATA_DIR.iterdir():
            print(f"  {item.name}")
else:
    print("\n[OK] All required data files found!")

## 2. Hyperparameters & Configuration

In [ ]:
# ============================================================================
# TRAINING CONFIGURATION
# ============================================================================

# Model
MODEL_ID = "facebook/mbart-large-50-many-to-many-mmt"

# mBART language codes (using Arabic as proxy for Akkadian)
SRC_LANG = "ar_AR"
TGT_LANG = "en_XX"

# Reproducibility
SEED = 42

# Sequence lengths
MAX_SOURCE_LENGTH = 256
MAX_TARGET_LENGTH = 256

# Batch size (adjust based on GPU memory)
# P100/T4 (16GB): batch_size=4, gradient_accumulation=4 -> effective=16
# If OOM, reduce batch_size to 2 and increase gradient_accumulation to 8
BATCH_SIZE = 4
GRADIENT_ACCUMULATION_STEPS = 4
EFFECTIVE_BATCH_SIZE = BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS

# Stage-specific settings
STAGE_CONFIG = {
    1: {
        "epochs": 5,
        "learning_rate": 1e-4,
        "warmup_steps": 500,
        "train_csv": DATA_DIR / "stage1_train.csv",
    },
    2: {
        "epochs": 3,
        "learning_rate": 5e-5,  # Lower LR for domain adaptation
        "warmup_steps": 300,
        "train_csv": DATA_DIR / "stage2_train.csv",
    },
    3: {
        "epochs": 10,
        "learning_rate": 3e-5,  # Even lower for fine-tuning
        "warmup_steps": 200,
        "train_csv": DATA_DIR / "stage3_train.csv",
        "val_csv": DATA_DIR / "stage3_val.csv",
        "early_stopping_patience": 3,
    },
}

print(f"Effective batch size: {EFFECTIVE_BATCH_SIZE}")
print(f"Model: {MODEL_ID}")

## 3. Utility Functions

In [ ]:
def set_seeds(seed: int = 42):
    """Set random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seeds(SEED)
print(f"Seeds set to {SEED}")

In [ ]:
def load_stage_data(stage: int, split: str = "train") -> pd.DataFrame:
    """Load data for a specific training stage."""
    config = STAGE_CONFIG[stage]
    
    if split == "train":
        csv_path = config["train_csv"]
    elif split == "val" and "val_csv" in config:
        csv_path = config["val_csv"]
    else:
        raise ValueError(f"Invalid split '{split}' for stage {stage}")
    
    df = pd.read_csv(csv_path)
    
    # Normalize column names
    if "transliteration" in df.columns:
        df = df.rename(columns={
            "transliteration": "transliteration_normalized",
            "translation": "translation_normalized",
        })
    elif "first_word_spelling" in df.columns:
        # Stage 2 uses different column names
        df = df.rename(columns={
            "first_word_spelling": "transliteration_normalized",
            "translation": "translation_normalized",
        })
    
    # Drop rows with null values
    initial_count = len(df)
    df = df.dropna(subset=["transliteration_normalized", "translation_normalized"])
    dropped = initial_count - len(df)
    if dropped > 0:
        print(f"  Dropped {dropped} rows with null values")
    
    return df.reset_index(drop=True)

# Test loading
for stage in [1, 2, 3]:
    if STAGE_CONFIG[stage]["train_csv"].exists():
        df = load_stage_data(stage)
        print(f"Stage {stage}: {len(df)} training samples")

In [ ]:
class TranslationDataset(Dataset):
    """PyTorch Dataset for translation pairs."""
    
    def __init__(self, df, tokenizer, max_source_length, max_target_length):
        self.df = df
        self.tokenizer = tokenizer
        self.max_source_length = max_source_length
        self.max_target_length = max_target_length
    
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        return {
            "source": row["transliteration_normalized"],
            "target": row["translation_normalized"],
        }


def collate_fn(batch, tokenizer, max_source_length, max_target_length):
    """Collate function for DataLoader."""
    sources = [item["source"] for item in batch]
    targets = [item["target"] for item in batch]
    
    # Tokenize
    model_inputs = tokenizer(
        sources,
        text_target=targets,
        max_length=max_source_length,
        padding="max_length",
        truncation=True,
        return_tensors="pt",
    )
    
    # Replace padding token id with -100 for labels
    labels = model_inputs["labels"]
    labels[labels == tokenizer.pad_token_id] = -100
    
    return {
        "input_ids": model_inputs["input_ids"],
        "attention_mask": model_inputs["attention_mask"],
        "labels": labels,
    }


def create_dataloader(stage, tokenizer, split="train"):
    """Create DataLoader for a training stage."""
    df = load_stage_data(stage, split)
    
    dataset = TranslationDataset(
        df, tokenizer, MAX_SOURCE_LENGTH, MAX_TARGET_LENGTH
    )
    
    collate = partial(
        collate_fn,
        tokenizer=tokenizer,
        max_source_length=MAX_SOURCE_LENGTH,
        max_target_length=MAX_TARGET_LENGTH,
    )
    
    return DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        collate_fn=collate,
        shuffle=(split == "train"),
    )

In [ ]:
class EarlyStopping:
    """Early stopping based on validation metric."""
    
    def __init__(self, patience: int = 3, min_delta: float = 0.0):
        self.patience = patience
        self.min_delta = min_delta
        self.counter = 0
        self.best_score = None
        self.early_stop = False
    
    def __call__(self, score: float) -> bool:
        """Check if training should stop. Returns True if should stop."""
        if self.best_score is None:
            self.best_score = score
            return False
        
        if score < self.best_score + self.min_delta:
            self.counter += 1
            print(f"  EarlyStopping: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True
                return True
        else:
            self.best_score = score
            self.counter = 0
        
        return False
    
    def is_best(self, score: float) -> bool:
        """Check if score is best so far."""
        return self.best_score is None or score >= self.best_score

## 4. Training Functions

In [ ]:
def train_stage(
    stage: int,
    model,
    tokenizer,
    device,
    checkpoint_from: Path = None,
):
    """
    Train a single stage of the curriculum learning pipeline.
    
    Args:
        stage: Training stage (1, 2, or 3)
        model: The model to train
        tokenizer: The tokenizer
        device: Training device
        checkpoint_from: Previous stage checkpoint (for stages 2 and 3)
    
    Returns:
        Path to the best checkpoint directory
    """
    print(f"\n{'='*60}")
    print(f"STAGE {stage} TRAINING")
    print(f"{'='*60}\n")
    
    config = STAGE_CONFIG[stage]
    epochs = config["epochs"]
    learning_rate = config["learning_rate"]
    warmup_steps = config["warmup_steps"]
    
    # Create output directory
    stage_output_dir = OUTPUT_DIR / f"stage{stage}_final"
    stage_output_dir.mkdir(parents=True, exist_ok=True)
    
    # Create dataloader
    print(f"Loading Stage {stage} training data...")
    train_loader = create_dataloader(stage, tokenizer, "train")
    print(f"Training batches: {len(train_loader)}")
    
    # Validation loader for Stage 3
    val_loader = None
    if stage == 3 and "val_csv" in config:
        val_loader = create_dataloader(stage, tokenizer, "val")
        print(f"Validation batches: {len(val_loader)}")
    
    # Setup optimizer and scheduler
    optimizer = AdamW(model.parameters(), lr=learning_rate)
    total_steps = len(train_loader) * epochs // GRADIENT_ACCUMULATION_STEPS
    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps,
    )
    
    # Early stopping for Stage 3
    early_stopping = None
    if stage == 3:
        patience = config.get("early_stopping_patience", 3)
        early_stopping = EarlyStopping(patience=patience)
    
    # Metrics log
    metrics_log = []
    best_checkpoint_dir = None
    global_step = 0
    
    # Training loop
    model.train()
    for epoch in range(epochs):
        epoch_loss = 0.0
        epoch_steps = 0
        optimizer.zero_grad()
        
        for step, batch in enumerate(train_loader):
            # Move batch to device
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)
            
            # Forward pass
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels,
            )
            loss = outputs.loss / GRADIENT_ACCUMULATION_STEPS
            loss.backward()
            
            epoch_loss += outputs.loss.item()
            epoch_steps += 1
            
            # Gradient accumulation step
            if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0:
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad()
                global_step += 1
        
        # Compute average loss
        avg_loss = epoch_loss / epoch_steps if epoch_steps > 0 else 0.0
        
        # Log metrics
        metrics = {
            "epoch": epoch + 1,
            "train_loss": avg_loss,
            "global_step": global_step,
        }
        
        print(f"[Epoch {epoch + 1}/{epochs}] train_loss: {avg_loss:.4f}")
        
        # Save checkpoint
        epoch_dir = stage_output_dir / f"epoch_{epoch + 1}"
        epoch_dir.mkdir(parents=True, exist_ok=True)
        model.save_pretrained(epoch_dir)
        tokenizer.save_pretrained(epoch_dir)
        
        # Save training state
        training_state = {
            "epoch": epoch,
            "global_step": global_step,
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
        }
        torch.save(training_state, epoch_dir / "training_state.pt")
        
        # Track best checkpoint
        if best_checkpoint_dir is None:
            best_checkpoint_dir = epoch_dir
        
        # Early stopping check for Stage 3 (use negative loss as score)
        if early_stopping is not None:
            # Using negative loss since early stopping expects higher = better
            score = -avg_loss
            if early_stopping.is_best(score):
                best_checkpoint_dir = epoch_dir
            if early_stopping(score):
                print(f"Early stopping triggered at epoch {epoch + 1}")
                break
        else:
            # For stages 1 and 2, use last epoch as best
            best_checkpoint_dir = epoch_dir
        
        metrics_log.append(metrics)
    
    # Save metrics log
    metrics_path = stage_output_dir / "train_metrics.json"
    with open(metrics_path, "w") as f:
        json.dump(metrics_log, f, indent=2)
    
    print(f"\nStage {stage} training complete!")
    print(f"Best checkpoint: {best_checkpoint_dir}")
    
    return best_checkpoint_dir

## 5. Run Training Pipeline

In [ ]:
# Use the device determined in the setup cell
device = DEVICE
print(f"Using device: {device}")

if device.type == 'cpu':
    print("\n[WARNING] Training on CPU will be significantly slower.")
    print("Estimated time: 10-20+ hours for full training.")
    print("Consider reducing epochs or using a smaller batch size.")

# Load initial model and tokenizer
print(f"\nLoading model: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_ID)

# Configure mBART tokenizer
if hasattr(tokenizer, "src_lang"):
    tokenizer.src_lang = SRC_LANG
if hasattr(tokenizer, "tgt_lang"):
    tokenizer.tgt_lang = TGT_LANG

model.to(device)
print(f"Model loaded successfully")

In [ ]:
# ============================================================================
# STAGE 1: Pre-training on General Corpus
# ============================================================================

stage1_checkpoint = train_stage(
    stage=1,
    model=model,
    tokenizer=tokenizer,
    device=device,
)

In [ ]:
# ============================================================================
# STAGE 2: Domain Adaptation (OARE)
# ============================================================================

# Load from Stage 1 checkpoint
print(f"\nLoading Stage 1 checkpoint: {stage1_checkpoint}")
tokenizer = AutoTokenizer.from_pretrained(stage1_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(stage1_checkpoint)

# Configure tokenizer
if hasattr(tokenizer, "src_lang"):
    tokenizer.src_lang = SRC_LANG
if hasattr(tokenizer, "tgt_lang"):
    tokenizer.tgt_lang = TGT_LANG

model.to(device)

stage2_checkpoint = train_stage(
    stage=2,
    model=model,
    tokenizer=tokenizer,
    device=device,
    checkpoint_from=stage1_checkpoint,
)

In [ ]:
# ============================================================================
# STAGE 3: Competition Fine-Tuning
# ============================================================================

# Load from Stage 2 checkpoint
print(f"\nLoading Stage 2 checkpoint: {stage2_checkpoint}")
tokenizer = AutoTokenizer.from_pretrained(stage2_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(stage2_checkpoint)

# Configure tokenizer
if hasattr(tokenizer, "src_lang"):
    tokenizer.src_lang = SRC_LANG
if hasattr(tokenizer, "tgt_lang"):
    tokenizer.tgt_lang = TGT_LANG

model.to(device)

stage3_checkpoint = train_stage(
    stage=3,
    model=model,
    tokenizer=tokenizer,
    device=device,
    checkpoint_from=stage2_checkpoint,
)

In [ ]:
# ============================================================================
# EXPORT FINAL MODEL
# ============================================================================

final_model_dir = OUTPUT_DIR / "submission_model"
final_model_dir.mkdir(parents=True, exist_ok=True)

print(f"\nExporting final model to: {final_model_dir}")

# Load best checkpoint from Stage 3
tokenizer = AutoTokenizer.from_pretrained(stage3_checkpoint)
model = AutoModelForSeq2SeqLM.from_pretrained(stage3_checkpoint)

# Save final model
model.save_pretrained(final_model_dir)
tokenizer.save_pretrained(final_model_dir)

# Write training summary
summary = {
    "model_id": MODEL_ID,
    "stage1_checkpoint": str(stage1_checkpoint),
    "stage2_checkpoint": str(stage2_checkpoint),
    "stage3_checkpoint": str(stage3_checkpoint),
    "final_model": str(final_model_dir),
    "completed_at": datetime.now(timezone.utc).isoformat(),
}

with open(final_model_dir / "training_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\n" + "="*60)
print("TRAINING COMPLETE!")
print("="*60)
print(f"\nFinal model saved to: {final_model_dir}")
print("\nNext steps:")
print("1. Run the inference notebook to generate predictions")
print("2. Submit your predictions to the competition")

## 6. Quick Test (Optional)

In [ ]:
# Test the trained model with a sample input
test_input = "a-na LUGAL be-li2-ia"

print(f"Test input: {test_input}")

# Load final model
tokenizer = AutoTokenizer.from_pretrained(final_model_dir)
model = AutoModelForSeq2SeqLM.from_pretrained(final_model_dir)
model.to(device)
model.eval()

# Generate translation
inputs = tokenizer(test_input, return_tensors="pt", padding=True)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_length=256,
        num_beams=4,
        early_stopping=True,
    )

translation = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(f"Translation: {translation}")